[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CCS-ZCU/EMLAP_ETL/blob/master/scripts/emlap-tokens-symbols.ipynb)

This Jupyter Notebook, which can be executed both locally or using a Cloud Computing service such as Google Colab, serves to demonstrate the potential of the computationally readable & morphologically annotated version of the EMLAP corpus.

It combines two publicly data files: one for EMLAP metadata, one for the morphologically annotated token data. These two can be mapped on each other using the individual document id (`no.` attribute in the metadata table, and `id` attribute in the tokens table). As this notebook demonsrates, these two files allow to proceed with incredible diverse types of analysis without necessity to download and parse multiple files.

This representation of the data is very similar to the data model used within the GreLa database, which however offers some additional advanced and efficient database-search functionalities.

In [2]:
import pandas as pd
import requests
import nltk
import re
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import matplotlib.pyplot as plt

In [29]:
try:
    import google_conf
    symbols_gsheet = google_conf.setup(sheet_url="https://docs.google.com/spreadsheets/d/1FMOpGA0W-HM3OnwXSahzUrb-UyR3tupmmFpGeJV3nBs/edit?usp=sharing", service_account_path="../../../ServiceAccountsKey.json")
except:
    print("Probably you are not working with the same setup as me (Vojtěch) when working locally. Therefore, the google sheet integration (uploading dataframes to a specific google sheet file) will not work for you. But this is not critical for the rest of the code.")

In [4]:
# first let's load our latest version of the metadata table
emlap_metadata = pd.read_csv("https://raw.githubusercontent.com/CCS-ZCU/EMLAP_ETL/refs/heads/master/data/emlap_metadata.csv", index_col=0, sep=";")
emlap_metadata.head(5)

,Unnamed: 0,working_title,filenames,no.,is_done,is_noscemus,if_noscemus_id,AUTHORSHIP,is_one_author,#if more than 1 author skip section and choose compendium below,...,CONTENTS,genre,subject,SOURCE OF FILE,link,source_of_file,origin_of_copy,other_notes,tokens_N,aurhor_wd
0,0,"Augurello, Chrysopoeia",100001_Augurello1515_Chrysopoeia_GB_Noscemus,100001,True,True,713324.0,NaN,True,NaN,...,NaN,didactic poem,alchemy,NaN,https://wiki.uibk.ac.at/noscemus/Chrysopoeia,GB,Noscemus,NaN,23225,NaN
1,1,"Pseudo-Lull, Secretis",100002_Pseudo-Lull1518_De secretis_naturae_MDZ...,100002,True,False,NaN,NaN,True,NaN,...,NaN,treatise,"alchemy, medicine",NaN,https://www.digitale-sammlungen.de/en/view/bsb...,MDZ,MBS,NaN,24696,NaN
2,2,"Pantheus, Ars Transmutatione",100003_Pantheus1518_Ars_Transmutationis_Metall...,100003,True,False,NaN,NaN,True,NaN,...,NaN,treatise,alchemy,NaN,https://www.google.co.uk/books/edition/Ars_Tra...,GB,BL,NaN,8683,NaN
3,3,"Anon, Vera alchemiae",100004_Anon1561_Verae_Alchemiae_MDZ_MBS,100004,True,False,NaN,NaN,True,NaN,...,NaN,"compendium, florilegium",alchemy,NaN,https://mdz-nbn-resolving.de/details:bsb10141168,MDZ,MBS,NaN,368660,NaN
4,4,"Pantheus, Voarchadumia",100005_Pantheus1530_Voarchadumia_ONB,100005,True,False,NaN,NaN,True,NaN,...,NaN,treatise,alchemy,NaN,https://data.onb.ac.at/rep/10588E49,ONB,ONB,NaN,21175,NaN


In [5]:
len(emlap_metadata)

100

In [6]:
# build a simple dict object mapping ids on titles
# useful for overview plots below
title_date = emlap_metadata.apply(lambda row: row["working_title"] + " ({})".format(str(row["date_publication"])), axis=1)
id_to_date_dict = dict(zip(emlap_metadata["no."].astype(str), emlap_metadata["date_publication"]))
id_to_title_dict = dict(zip(emlap_metadata["no."].astype(str), title_date))
id_to_title_dict

{'100001': 'Augurello, Chrysopoeia (1515)',
 '100002': 'Pseudo-Lull, Secretis (1518)',
 '100003': 'Pantheus, Ars Transmutatione (1518)',
 '100004': 'Anon, Vera alchemiae (1561)',
 '100005': 'Pantheus, Voarchadumia (1530)',
 '100006': 'Savonarola, De arte conficiendi aquam vitae (1532)',
 '100007': 'Anon, Rosarium philosophorum (1550)',
 '100008': 'Severinus, Epistola (1571)',
 '100009': 'Vegius, Inter inferiora corpora (1518)',
 '100010': 'Bracesco, De alchimia dialogi duo (1548)',
 '100011': 'Anon, De alchemia (1541)',
 '100012': 'Gessner, Euonymus (1552)',
 '100013': 'Ulstadt, Coelum (1525)',
 '100014': 'Toxites, Spongia (1567)',
 '100015': 'Gessner, Euonymus II (1569)',
 '100016': 'Bonus, Pretiosa margarita novella (1546)',
 '100017': 'Bodenstein, Isagoge (1559)',
 '100018': 'Trevisan, Peri chemeias (1567)',
 '100019': 'Ulstadt, De epidemia (1526)',
 '100020': 'Dorn, Artificii chymistici (1569)',
 '100021': 'Dorn, Clavis (1567)',
 '100022': 'Anon, De alchimia opuscula  (1550)',
 '10

In [7]:
# now load full table of morphologically annotated tokens data table directly from the server.
# This is a complete morphological annotated version of the dataset in one file, which can be used for any sort of down stream computational text analysis.
emlap_tokens_df = pd.read_parquet("https://ccs-lab.zcu.cz/emlap_corpus_public/emlap_tokens_df.parquet")
emlap_tokens_df.shape # let's inspect its shape (N rows x N columns

(6477016, 13)

In [8]:
# a random look somewhere to get an idea of the structure
emlap_tokens_df[100000:100010]

,token_text,lemma,pos,ref,char_start,char_end,id,sent_idx,sent_len,page,textblock,tag,blocktype
100000,magni,magnus,ADJ,"{'blocktype': 'text', 'page': [4], 'tag': '', ...",88,93,100014,24,129,[4],[15],,text
100001,",",",",PUNCT,"{'blocktype': 'text', 'page': [4], 'tag': '', ...",93,94,100014,24,129,[4],[15],,text
100002,Duxisti,duco,VERB,"{'blocktype': 'text', 'page': [4], 'tag': '', ...",95,102,100014,24,129,[4],[16],,text
100003,tanto,tantus,ADV,"{'blocktype': 'text', 'page': [4], 'tag': '', ...",103,108,100014,24,129,[4],[16],,text
100004,non,non,PART,"{'blocktype': 'text', 'page': [4], 'tag': '', ...",109,112,100014,24,129,[4],[16],,text
100005,tibi,tu,PRON,"{'blocktype': 'text', 'page': [4], 'tag': '', ...",113,117,100014,24,129,[4],[16],,text
100006,turpe,turpis,ADJ,"{'blocktype': 'text', 'page': [4], 'tag': '', ...",118,123,100014,24,129,[4],[16],,text
100007,uiro,uir,NOUN,"{'blocktype': 'text', 'page': [4], 'tag': '', ...",124,128,100014,24,129,[4],[16],,text
100008,.,.,PUNCT,"{'blocktype': 'text', 'page': [4], 'tag': '', ...",128,129,100014,24,129,[4],[16],,text
100009,Nil,nihil,PRON,"{'blocktype': 'text', 'page': [4], 'tag': '', ...",0,3,100014,25,18,[4],[17],,text


Explantion of individual attributes:

Basic explantion of individual columns/attributes:
* `token_text`: the token as it appears in the text, after some basic precleaning (character normalization etc.)
* `lemma`: a dictionary-entry-like form of the token, assigned automatically within a morphological analysis pipeline (*latinCy* model for Python *spaCy* library)
* `pos` : standard part-of-speech tag,  assigned automatically  within the morphological analysis pipeline
* `ref` : additional metadata for each token, especially concerning its location within the source (see below)
* `char_start` : position of the opening character of the token within the source sentence raw text - allows to navigate to specific token while being able to disambiguate multiple occurrences of the same token within a sentence (especially useful for contextual embeddings from GreLa)
* `char_end` : position of the ending character of the token within the source sentence raw text
* `id` : numerical identifier of the work within EMLAP - can be used to obtain any additional work-level metadata without duplicating them here
* `sent_idx` : positional index of token's source sentence within the source text. Sentence division assigned automatically  within the morphological analysis pipeline. When needed, allows to render  sentence-level text.
* `sent_len` : N of characters of the source sentence.
* `page` : page index (*0-based indexing*) from which the token comes from. Typically one number, but formatted as a list for machine readiblity, since sometimes a token crosses page boundaries (via hyphen, which we removed).
* `textblock` : textblock (usually a line) index (*in 0-based indexing*) from which the token comes from. Typically one number, but formatted as a list for machine readiblity, since sometimes a token crosses textblock boundaries (via hyphen, which we removed).
* `tag` : a tag assigned manually by human annotaters during the OCR data curation.
* `blocktype` : a manually (in case of margins) or automatically (in case of titles) assigned type of the text content. Possible values are "margin", "header" or "text" (textblocks classified as "header" or "footer" were removed during the cleaning phase).


In [9]:
# combining these columns, we can for instance count the frequencies of most common words, but filtered by specified "part-of-speech"
emlap_tokens_df[emlap_tokens_df["pos"].isin(["NOUN", "ADJ", "VERB", "PROPN"])]["lemma"].value_counts().head(20)

lemma
aqua        29613
facio       26164
possum      25180
natura      22679
dico        21829
corpus      20822
ignis       18172
habeo       16962
res         16163
pars        13625
spiritus    12795
aurum       11313
materia     11162
ars         11116
uideo       10647
terra       10400
argentum    10381
lapis       10349
oleum        9972
modus        9604
Name: count, dtype: int64

In [10]:
# we can also inspect distribution of individual POS tags.
emlap_tokens_df["pos"].value_counts()

pos
PUNCT    1434275
NOUN     1358905
VERB      915291
ADJ       489283
ADP       431607
ADV       396326
DET       330822
PRON      235866
SCONJ     204660
CCONJ     171060
AUX       157958
PROPN     141078
PART      113359
NUM        77330
SYM         8670
X           6592
INTJ        3934
Name: count, dtype: int64

In [11]:
# we can spot that there is our own manually created pos-tag "SYM", which is used for tokens manually tagged as symbol:

In [12]:
# we can create a table of all tokens tagged as symbols
emlap_symbols_tokens_df = emlap_tokens_df[emlap_tokens_df["pos"]=="SYM"]
len(emlap_symbols_tokens_df)

8670

In [13]:
emlap_symbols_tokens_df.value_counts("lemma")

lemma
y         2989
recipe    1464
z         1422
lb         936
scr        288
          ... 
♈            1
♉            1
♊            1
31505        1
♌            1
Name: count, Length: 134, dtype: int64

In [16]:
##adding a concordance
token_texts = emlap_tokens_df["token_text"].values
doc_ids = emlap_tokens_df["id"].values

In [17]:
def get_concordance(idx, context_size=10):
    target_id = doc_ids[idx]

    # Get preceding tokens (up to context_size)
    left_tokens = []
    for i in range(idx - 1, max(0, idx - context_size - 1), -1):
        if doc_ids[i] != target_id:
            break
        left_tokens.append(token_texts[i])
    left_tokens.reverse()

    # Get following tokens (up to context_size)
    right_tokens = []
    for i in range(idx + 1, min(len(doc_ids), idx + context_size + 1)):
        if doc_ids[i] != target_id:
            break
        right_tokens.append(token_texts[i])
    return " ".join(left_tokens), " ".join(right_tokens)

In [19]:
concordances = emlap_symbols_tokens_df.index.map(lambda idx: get_concordance(idx))

In [20]:
emlap_symbols_tokens_df["concordance_left"] = [c[0] for c in concordances]
emlap_symbols_tokens_df["concordance_right"] = [c[1] for c in concordances]

/tmp/ipykernel_2110510/2318211313.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  emlap_symbols_tokens_df["concordance_left"] = [c[0] for c in concordances]
/tmp/ipykernel_2110510/2318211313.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  emlap_symbols_tokens_df["concordance_right"] = [c[1] for c in concordances]


In [21]:
##lemma-based concordance with filtered POS tags
lemmas = emlap_tokens_df["lemma"].values
pos_tags = emlap_tokens_df["pos"].values

def get_concordance_lemma(idx, context_size=10):
    target_id = doc_ids[idx]
    pos_filter = {"NOUN", "ADJ", "VERB", "PROPN"}

    # Get preceding lemmas (only content words)
    left_lemmas = []
    for i in range(idx - 1, max(0, idx - context_size - 1), -1):
        if doc_ids[i] != target_id:
            break
        if pos_tags[i] in pos_filter:
            left_lemmas.append(lemmas[i])
    left_lemmas.reverse()

    # Get following lemmas (only content words)
    right_lemmas = []
    for i in range(idx + 1, min(len(doc_ids), idx + context_size + 1)):
        if doc_ids[i] != target_id:
            break
        if pos_tags[i] in pos_filter:
            right_lemmas.append(lemmas[i])

    return " ".join(left_lemmas), " ".join(right_lemmas)


In [22]:
concordances_lemma = emlap_symbols_tokens_df.index.map(lambda idx: get_concordance_lemma(idx))
emlap_symbols_tokens_df["concordance_left_lemma"] = [c[0] for c in concordances_lemma]
emlap_symbols_tokens_df["concordance_right_lemma"] = [c[1] for c in concordances_lemma]

/tmp/ipykernel_2110510/224408355.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  emlap_symbols_tokens_df["concordance_left_lemma"] = [c[0] for c in concordances_lemma]
/tmp/ipykernel_2110510/224408355.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  emlap_symbols_tokens_df["concordance_right_lemma"] = [c[1] for c in concordances_lemma]


In [23]:
emlap_symbols_tokens_df.sample(10, random_state=42)

,token_text,lemma,pos,ref,char_start,char_end,id,sent_idx,sent_len,page,textblock,tag,blocktype,concordance_left,concordance_right,concordance_left_lemma,concordance_right_lemma
1594300,◉,recipe,SYM,"{'blocktype': 'text', 'page': [156], 'tag': 'S...",23,24,100086,2667,32,[156],[5],S,text,& materia sublimabitur christalli instar . Si ...,de etc . Istius pulueris uires sunt maximae . ...,materia sublimo christallus instar multiplico ...,puluis uis magnus febris
4703072,◉,uenus,SYM,"{'blocktype': 'text', 'page': [24], 'tag': 'S'...",10,11,100027,215,73,[24],[13],S,text,clangorem . ◉ ◉ & ◉ . Habet ◉ a,"◉ & ◉ coagulationis mensuram , & malleationem ◉ ◉",clangor habeo,coagulatio mensura malleatio
1716706,◉,scr,SYM,"{'blocktype': 'text', 'page': [354], 'tag': 'S...",219,220,100028,5707,221,[354],[20],S,text,"duos , cum pauco uino albo , & tartari contriti",. ii . uel iii . bibantur . Recentem uero,paucus uinum albus tarto contero,bibo recens
6024812,◉,z,SYM,"{'blocktype': 'text', 'page': [93], 'tag': 'S'...",44,45,100015,975,50,[93],[9],S,text,"◉ ii . fellis perdicum , phasianorum & galloru...","iii . mellis ◉ . succi foeniculi , succi euphr...",fel perdix phasianus gallus anus,mel succus foeniculum succus euphragia
1740683,◉,z,SYM,"{'blocktype': 'text', 'page': [480], 'tag': 'S...",20,21,100028,8087,22,[480],[4],S,text,. ◉ . Iridis ◉ . iiii . Myrrhae Troglitidis,. ii . Piperis ◉ . i . Seselis ◉,irido Myrrha troglitidus,piper eo seselis
1123037,◉,cancer,SYM,"{'blocktype': 'text', 'page': [97], 'tag': 'S'...",32,33,100085,7270,77,[97],[18],S,text,◉ ◉ ◉ ◉ ◉ ◉ ◉ ◉ ◉ ◉,◉ ◉ ◉ ◉ ◉ ◉ ◉ quae teutonum more,,teutones mos
44376,◉,y,SYM,"{'blocktype': 'text', 'page': [227], 'tag': 'S...",0,1,100044,6602,2,[227],[9],S,text,mixtionem facilius admittant . postremo addas ...,. i . β . omnia permisceantur . hoc unguento,mixtio admitto addo Aristolochie rotundus pulu...,io ἐὕω permisceo unguentum
34420,◉,lb,SYM,"{'blocktype': 'text', 'page': [177], 'tag': 'S...",0,1,100044,5100,2,[177],[5],S,text,. Uitrioli . ◉ . i . Tartari calcinati .,. ◉ . Salis fusi . ◉ . iii .,uitriolum eo tartarus calcinor,salis fundo
20730,◉,y,SYM,"{'blocktype': 'text', 'page': [105], 'tag': 'S...",0,1,100044,3101,2,[105],[9],S,text,Aliud mundificatiuum oppodeltoch dictum . ◉ . ...,. i . Aquae culiculae . ◉ . i .,mundificatiuus oppodeltoch dico operimene fixus,io aquus culicula io
445996,◉,recipe,SYM,"{'blocktype': 'text', 'page': [191], 'tag': 'S...",4,5,100043,2918,42,[191],[21],S,text,. i . excipiantur melle & oui luteo : uel,sarcocollae in lacte solutae ◉ . iii . mastich...,io excipio mel ouum luteus,sarcocolla lac soluo mastiche


In [26]:
emlap_symbols_tokens_df["working_title"] = emlap_symbols_tokens_df["id"].apply(lambda x: id_to_title_dict.get(x))

/tmp/ipykernel_2110510/1047514375.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  emlap_symbols_tokens_df["working_title"] = emlap_symbols_tokens_df["id"].apply(lambda x: id_to_title_dict.get(x))


In [27]:
emlap_symbols_tokens_df.head(5)

,token_text,lemma,pos,ref,char_start,char_end,id,sent_idx,sent_len,page,textblock,tag,blocktype,concordance_left,concordance_right,concordance_left_lemma,concordance_right_lemma,working_title
4025,◉,recipe,SYM,"{'blocktype': 'text', 'page': [19], 'tag': 'S'...",0,1,100044,451,2,[19],[8],S,text,morborum methodum ordine alphabetico continens...,. Auripigmenti ◉ . iii . Aluminis calcinati ◉ .,morbus methodus ordo alphabeticus contineo aes...,auripigmentum alumen calcino,"Dorn, Theophrasti Germani (1578)"
4028,◉,y,SYM,"{'blocktype': 'text', 'page': [19], 'tag': 'S'...",13,14,100044,452,20,[19],[8],S,text,alphabetico continens . A Aestiomenum cura . ◉...,. iii . Aluminis calcinati ◉ . ui . Reduc,alphabeticus contineo aestiomenus cura auripig...,alumen calcino uis reduco,"Dorn, Theophrasti Germani (1578)"
4034,◉,y,SYM,"{'blocktype': 'text', 'page': [19], 'tag': 'S'...",19,20,100044,453,21,[19],[9],S,text,. ◉ . Auripigmenti ◉ . iii . Aluminis calcinati,. ui . Reduc & fiat alcali . Addi possunt,auripigmentum alumen calcino,uis reduco fio alcalis Addi possum,"Dorn, Theophrasti Germani (1578)"
4049,◉,z,SYM,"{'blocktype': 'text', 'page': [19], 'tag': 'S'...",20,21,100044,457,22,[19],[12],S,text,& fiat alcali . Addi possunt . Ad huius praepa...,. ui . Liquoris mumiae . ◉ . ii .,fio alcalis Addi possum praeparo,uis liquor mumia,"Dorn, Theophrasti Germani (1578)"
4056,◉,z,SYM,"{'blocktype': 'text', 'page': [19], 'tag': 'S'...",0,1,100044,460,2,[19],[13],S,text,Ad huius praeparati ◉ . ui . Liquoris mumiae .,. ii . Olei rosacei . ◉ . i .,praeparo uis liquor mumia,oleum rosaceus eo,"Dorn, Theophrasti Germani (1578)"


In [28]:
google_conf.set_with_dataframe(symbols_gsheet.add_worksheet("emlap_symbols_tokens_df", 1,1), emlap_symbols_tokens_df)